# GLA State Extraction - Testing and Verification

This notebook tests the state extraction mechanism for GLA (Gated Linear Attention) and compares different extraction methods.

**Goals:**
1. Load a GLA model (e.g., fla-hub/gla-1.3B-100B)
2. Compare 3 extraction methods:
   - `extract_final_states` - Final state only (single forward pass)
   - `extract_incremental_states_dumb_rerunning` - All positions (O(N²) - slow)
   - `extract_incremental_states_single_pass` - All positions (O(N) - efficient)
3. Verify correctness by comparing results
4. Measure and compare performance


In [ ]:
import sys
import os
import time
import torch
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

try:
    import google.colab
    IN_COLAB = True
    print("Running in Google Colab")
except:
    IN_COLAB = False
    print("Running locally")

if IN_COLAB:
    if not os.path.exists('state-games'):
        print("Cloning repository...")
        !git clone https://github.com/idoavnir-uni/state-games.git
        print("Repository cloned!")
    
    os.chdir('state-games')
    print(f"Current directory: {os.getcwd()}")
    
    print("\nInstalling dependencies...")
    %pip install -q torch>=2.0.0 transformers>=4.30.0 huggingface_hub numpy pandas matplotlib einops h5py scikit-learn
    
    print("\nInstalling Flash Linear Attention library...")
    %pip install -q git+https://github.com/sustcsonglin/flash-linear-attention.git
    
    print("\nDependencies installed!")

if IN_COLAB:
    sys.path.insert(0, '/content/state-games')
else:
    sys.path.insert(0, os.path.abspath('..'))

from models.load_gla import load_gla_model, get_model_config, print_model_structure
from models.state_extractor_gla import GLAStateExtractor

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("Setup complete!")


## 1. Load Model and Configuration


In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

if device == "cpu":
    print("WARNING: Running on CPU. This will be very slow for large models.")
    print("Consider running on a GPU or using a smaller model for testing.")


In [ ]:
print("Loading GLA model...")
model, tokenizer = load_gla_model(
    model_name="fla-hub/gla-1.3B-100B",
    device=device,
    torch_dtype=torch.bfloat16
)


In [ ]:
config = get_model_config(model)

print("\n=== Key Configuration ===")
print(f"Number of layers: {config.get('num_layers', 'Unknown')}")
print(f"Number of heads: {config.get('num_heads', 'Unknown')}")
print(f"Hidden size: {config.get('hidden_size', 'Unknown')}")
print(f"Vocabulary size: {config.get('vocab_size', 'Unknown')}")
print(f"Max sequence length: {config.get('max_seq_len', 'Unknown')}")
print(f"Expand K: {config.get('expand_k', 'Unknown')}")
print(f"Expand V: {config.get('expand_v', 'Unknown')}")
print(f"Attention mode: {config.get('attn_mode', 'Unknown')}")
print(f"Use short conv: {config.get('use_short_conv', 'Unknown')}")


In [ ]:
print_model_structure(model, max_depth=3)


## 2. Initialize State Extractor and Prepare Test Input


In [ ]:
extractor = GLAStateExtractor(model, verbose=True)

test_text = "The quick brown fox jumps over the lazy dog."
print(f"Input text: '{test_text}'")

inputs = tokenizer(test_text, return_tensors="pt")
input_ids = inputs.input_ids.to(device)

print(f"Token IDs shape: {input_ids.shape}")
print(f"Tokens: {tokenizer.convert_ids_to_tokens(input_ids[0])}")
print(f"\nState extractor ready")


In [ ]:
with torch.no_grad():
    outputs = model(input_ids, use_cache=True)

print(f"Outputs type: {type(outputs)}")
print(f"past_key_values type: {type(outputs.past_key_values)}")
print(f"Number of layers: {len(outputs.past_key_values)}")

print(f"\nFirst layer cache:")
first_layer = outputs.past_key_values[0]
print(f"Type: {type(first_layer)}")
print(f"Keys: {first_layer.keys() if isinstance(first_layer, dict) else 'Not a dict'}")

if "recurrent_state" in first_layer:
    print(f"\nrecurrent_state shape: {first_layer['recurrent_state'].shape}")
    print(f"recurrent_state dtype: {first_layer['recurrent_state'].dtype}")
    
print(f"\nAll available keys in first layer:")
for key, value in first_layer.items():
    if isinstance(value, torch.Tensor):
        print(f"  {key}: shape={value.shape}, dtype={value.dtype}")
    elif isinstance(value, tuple) and len(value) > 0:
        print(f"  {key}: tuple with {len(value)} elements")


## 3. Compare Extraction Methods

We compare three extraction methods:
1. **`extract_final_states`** - Gets only the final state (single forward pass)
2. **`extract_incremental_states_dumb_rerunning`** - Gets all intermediate states (O(N²) - runs full prefix for each position)
3. **`extract_incremental_states_single_pass`** - Gets all intermediate states (O(N) - efficient incremental processing)


In [ ]:
print("=" * 60)
print("METHOD 1: extract_final_states (final state only)")
print("=" * 60)

start_time = time.time()
final_states = extractor.extract_final_states(input_ids)
time_method1 = time.time() - start_time

print(f"\nTime: {time_method1:.4f}s")
print(f"Number of layers: {len(final_states)}")
if final_states:
    first_layer_state = final_states[0]
    print(f"State shape per layer: {first_layer_state.shape}")


In [ ]:
print("=" * 60)
print("METHOD 2: extract_incremental_states_dumb_rerunning (O(N²) - slow)")
print("=" * 60)

start_time = time.time()
incremental_states = extractor.extract_incremental_states_dumb_rerunning(input_ids)
time_method2 = time.time() - start_time

print(f"\nTime: {time_method2:.4f}s")
print(f"Number of positions: {len(incremental_states)}")
first_pos_states = incremental_states[1]
print(f"Number of layers per position: {len(first_pos_states)}")
print(f"State shape at each position: {first_pos_states[0].shape}")


In [ ]:
print("=" * 60)
print("METHOD 3: extract_incremental_states_single_pass (O(N) - efficient)")
print("=" * 60)

start_time = time.time()
single_pass_states = extractor.extract_incremental_states_single_pass(input_ids)
time_method3 = time.time() - start_time

print(f"\nTime: {time_method3:.4f}s")
print(f"Number of positions: {len(single_pass_states)}")
first_pos_states_sp = single_pass_states[1]
print(f"Number of layers per position: {len(first_pos_states_sp)}")
print(f"State shape at each position: {first_pos_states_sp[0].shape}")


## 4. Compare Results and Verify Correctness

This section verifies that the efficient method produces the same results as the slow method.


In [ ]:
print("=" * 60)
print("TIMING COMPARISON")
print("=" * 60)
seq_len = input_ids.shape[1]
print(f"\nSequence length: {seq_len} tokens")
print(f"\nMethod 1 (final state only):    {time_method1:.4f}s")
print(f"Method 2 (incremental, O(N²)):  {time_method2:.4f}s")
print(f"Method 3 (single-pass, O(N)):   {time_method3:.4f}s")
print(f"\nSpeedup (Method 3 vs Method 2): {time_method2 / time_method3:.2f}x")


In [ ]:
print("=" * 60)
print("SHAPE VERIFICATION")
print("=" * 60)

seq_len = input_ids.shape[1]

print("\n1. Comparing shapes: Method 1 (final) vs Method 2 (last position):")
shapes_match_1_vs_2 = True
for layer_idx in final_states.keys():
    state_m1 = final_states[layer_idx]
    state_m2 = incremental_states[seq_len][layer_idx]
    if state_m1.shape != state_m2.shape:
        shapes_match_1_vs_2 = False
        print(f"   Layer {layer_idx}: MISMATCH - {state_m1.shape} vs {state_m2.shape}")
print(f"   All shapes match: {shapes_match_1_vs_2}")

print("\n2. Comparing shapes: Method 1 (final) vs Method 3 (last position):")
shapes_match_1_vs_3 = True
for layer_idx in final_states.keys():
    state_m1 = final_states[layer_idx]
    state_m3 = single_pass_states[seq_len][layer_idx]
    if state_m1.shape != state_m3.shape:
        shapes_match_1_vs_3 = False
        print(f"   Layer {layer_idx}: MISMATCH - {state_m1.shape} vs {state_m3.shape}")
print(f"   All shapes match: {shapes_match_1_vs_3}")

print("\n3. Comparing shapes: Method 2 vs Method 3 (all positions):")
shapes_match_2_vs_3 = True
shape_mismatches = []
for pos in incremental_states.keys():
    for layer_idx in incremental_states[pos].keys():
        state_m2 = incremental_states[pos][layer_idx]
        state_m3 = single_pass_states[pos][layer_idx]
        if state_m2.shape != state_m3.shape:
            shapes_match_2_vs_3 = False
            shape_mismatches.append((pos, layer_idx, state_m2.shape, state_m3.shape))

if shape_mismatches:
    print(f"   Found {len(shape_mismatches)} shape mismatches:")
    for pos, layer_idx, shape_m2, shape_m3 in shape_mismatches[:5]:
        print(f"     Position {pos}, Layer {layer_idx}: {shape_m2} vs {shape_m3}")
else:
    print(f"   All shapes match: {shapes_match_2_vs_3}")


In [ ]:
print("=" * 60)
print("VALUE VERIFICATION WITH HISTOGRAMS")
print("=" * 60)

seq_len = input_ids.shape[1]

print("\n1. Comparing values: Method 1 (final) vs Method 2 (last position):")
all_diffs_1_vs_2 = []
for layer_idx in final_states.keys():
    state_m1 = final_states[layer_idx]
    state_m2 = incremental_states[seq_len][layer_idx]
    diff = (state_m1 - state_m2).abs()
    all_diffs_1_vs_2.append(diff.flatten())
    
all_diffs_1_vs_2 = torch.cat(all_diffs_1_vs_2).numpy()
max_diff_1_vs_2 = all_diffs_1_vs_2.max()
mean_diff_1_vs_2 = all_diffs_1_vs_2.mean()
median_diff_1_vs_2 = np.median(all_diffs_1_vs_2)

print(f"   Max diff: {max_diff_1_vs_2:.2e}")
print(f"   Mean diff: {mean_diff_1_vs_2:.2e}")
print(f"   Median diff: {median_diff_1_vs_2:.2e}")
print(f"   Values match (rtol=1e-4, atol=1e-6): {max_diff_1_vs_2 < 1e-4}")

print("\n2. Comparing values: Method 1 (final) vs Method 3 (last position):")
all_diffs_1_vs_3 = []
for layer_idx in final_states.keys():
    state_m1 = final_states[layer_idx]
    state_m3 = single_pass_states[seq_len][layer_idx]
    diff = (state_m1 - state_m3).abs()
    all_diffs_1_vs_3.append(diff.flatten())
    
all_diffs_1_vs_3 = torch.cat(all_diffs_1_vs_3).numpy()
max_diff_1_vs_3 = all_diffs_1_vs_3.max()
mean_diff_1_vs_3 = all_diffs_1_vs_3.mean()
median_diff_1_vs_3 = np.median(all_diffs_1_vs_3)

print(f"   Max diff: {max_diff_1_vs_3:.2e}")
print(f"   Mean diff: {mean_diff_1_vs_3:.2e}")
print(f"   Median diff: {median_diff_1_vs_3:.2e}")
print(f"   Values match (rtol=1e-4, atol=1e-6): {max_diff_1_vs_3 < 1e-4}")

print("\n3. Comparing values: Method 2 vs Method 3 (all positions):")
all_diffs_2_vs_3 = []
for pos in incremental_states.keys():
    for layer_idx in incremental_states[pos].keys():
        state_m2 = incremental_states[pos][layer_idx]
        state_m3 = single_pass_states[pos][layer_idx]
        diff = (state_m2 - state_m3).abs()
        all_diffs_2_vs_3.append(diff.flatten())

all_diffs_2_vs_3 = torch.cat(all_diffs_2_vs_3).numpy()
max_diff_2_vs_3 = all_diffs_2_vs_3.max()
mean_diff_2_vs_3 = all_diffs_2_vs_3.mean()
median_diff_2_vs_3 = np.median(all_diffs_2_vs_3)

print(f"   Max diff: {max_diff_2_vs_3:.2e}")
print(f"   Mean diff: {mean_diff_2_vs_3:.2e}")
print(f"   Median diff: {median_diff_2_vs_3:.2e}")
print(f"   Values match (rtol=1e-4, atol=1e-6): {max_diff_2_vs_3 < 1e-4}")


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

axes[0, 0].hist(np.log10(all_diffs_1_vs_2 + 1e-20), bins=50, color='#3498db', edgecolor='black', alpha=0.7)
axes[0, 0].set_xlabel('Log10(Absolute Difference)', fontsize=11)
axes[0, 0].set_ylabel('Frequency', fontsize=11)
axes[0, 0].set_title('Method 1 vs Method 2 (Final State)\nDifference Distribution', fontsize=12, fontweight='bold')
axes[0, 0].axvline(np.log10(1e-4), color='red', linestyle='--', linewidth=2, label='rtol=1e-4 threshold')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

axes[0, 1].hist(np.log10(all_diffs_1_vs_3 + 1e-20), bins=50, color='#2ecc71', edgecolor='black', alpha=0.7)
axes[0, 1].set_xlabel('Log10(Absolute Difference)', fontsize=11)
axes[0, 1].set_ylabel('Frequency', fontsize=11)
axes[0, 1].set_title('Method 1 vs Method 3 (Final State)\nDifference Distribution', fontsize=12, fontweight='bold')
axes[0, 1].axvline(np.log10(1e-4), color='red', linestyle='--', linewidth=2, label='rtol=1e-4 threshold')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

axes[1, 0].hist(np.log10(all_diffs_2_vs_3 + 1e-20), bins=50, color='#e74c3c', edgecolor='black', alpha=0.7)
axes[1, 0].set_xlabel('Log10(Absolute Difference)', fontsize=11)
axes[1, 0].set_ylabel('Frequency', fontsize=11)
axes[1, 0].set_title('Method 2 vs Method 3 (All Positions)\nDifference Distribution', fontsize=12, fontweight='bold')
axes[1, 0].axvline(np.log10(1e-4), color='red', linestyle='--', linewidth=2, label='rtol=1e-4 threshold')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

stats_text = f"""Summary Statistics:

Method 1 vs 2:
  Max: {max_diff_1_vs_2:.2e}
  Mean: {mean_diff_1_vs_2:.2e}
  Median: {median_diff_1_vs_2:.2e}

Method 1 vs 3:
  Max: {max_diff_1_vs_3:.2e}
  Mean: {mean_diff_1_vs_3:.2e}
  Median: {median_diff_1_vs_3:.2e}

Method 2 vs 3:
  Max: {max_diff_2_vs_3:.2e}
  Mean: {mean_diff_2_vs_3:.2e}
  Median: {median_diff_2_vs_3:.2e}
"""

axes[1, 1].text(0.1, 0.5, stats_text, fontsize=11, verticalalignment='center', 
                family='monospace', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.3))
axes[1, 1].axis('off')
axes[1, 1].set_title('Difference Statistics', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

print("\nInterpretation:")
print("- Histograms show log10 of absolute differences")
print("- Red dashed line shows the rtol=1e-4 threshold")
print("- Values to the left of the line are within tolerance")
print("- Most differences should be at machine precision (~1e-16 for float64, ~1e-7 for float32)")


In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

methods = ['Method 1\n(final only)', 'Method 2\n(incremental O(N²))', 'Method 3\n(single-pass O(N))']
times = [time_method1, time_method2, time_method3]
colors = ['#2ecc71', '#e74c3c', '#3498db']

bars = ax.bar(methods, times, color=colors, edgecolor='black', linewidth=1.5)

for bar, t in zip(bars, times):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01, 
            f'{t:.3f}s', ha='center', va='bottom', fontsize=12, fontweight='bold')

ax.set_ylabel('Time (seconds)', fontsize=12)
ax.set_title(f'GLA State Extraction Methods Comparison\n(Sequence length: {seq_len} tokens)', fontsize=14)
ax.set_ylim(0, max(times) * 1.2)

plt.tight_layout()
plt.show()

print(f"\n=== FINAL SUMMARY ===")
print(f"Timing:")
print(f"  - For final state only: Use Method 1 (fastest - {time_method1:.3f}s)")
print(f"  - For all intermediate states: Use Method 3 (single-pass - {time_method3:.3f}s)")
print(f"  - Method 3 is {time_method2/time_method3:.1f}x faster than Method 2")
print(f"\nAccuracy:")
print(f"  - Method 1 vs 2: max diff = {max_diff_1_vs_2:.2e}")
print(f"  - Method 1 vs 3: max diff = {max_diff_1_vs_3:.2e}")
print(f"  - Method 2 vs 3: max diff = {max_diff_2_vs_3:.2e}")
if max_diff_2_vs_3 < 1e-4:
    print(f"  ✓ All methods produce equivalent results (within tolerance)")
else:
    print(f"  ⚠ Methods may have numerical differences above tolerance")


## 5. State Usage Verification Tests

These tests verify that the single-pass method correctly accumulates and uses state.


In [ ]:
print("=" * 60)
print("TEST 1: State Usage Sanity Check")
print("=" * 60)
print("\nVerify that the model actually uses the accumulated state.")
print("Compare inference on last token WITH state vs WITHOUT state.")

import copy

seq_len = input_ids.shape[1]
last_token = input_ids[:, -1:]

with torch.no_grad():
    past_key_values = None
    for pos in range(seq_len - 1):
        current_ids = input_ids[:, pos:pos+1]
        outputs = model(current_ids, past_key_values=past_key_values, use_cache=True)
        past_key_values = copy.deepcopy(outputs.past_key_values)
    
    outputs_with_state = model(last_token, past_key_values=past_key_values, use_cache=True)
    logits_with_state = outputs_with_state.logits
    
    outputs_without_state = model(last_token, use_cache=True)
    logits_without_state = outputs_without_state.logits

logits_diff = (logits_with_state - logits_without_state).abs()
max_logit_diff = logits_diff.max().item()
mean_logit_diff = logits_diff.mean().item()

print(f"\nLogits shape: {logits_with_state.shape}")
print(f"\nLogit differences (WITH state vs WITHOUT state):")
print(f"  Max diff:  {max_logit_diff:.4f}")
print(f"  Mean diff: {mean_logit_diff:.4f}")

pred_with_state = logits_with_state.argmax(dim=-1).item()
pred_without_state = logits_without_state.argmax(dim=-1).item()

print(f"\nPredicted next token:")
print(f"  WITH state:    {pred_with_state} -> '{tokenizer.decode([pred_with_state])}'")
print(f"  WITHOUT state: {pred_without_state} -> '{tokenizer.decode([pred_without_state])}'")

if max_logit_diff > 0.1:
    print(f"\n✓ PASS: Logits are DIFFERENT - state is being used correctly!")
else:
    print(f"\n✗ FAIL: Logits are nearly identical - state may not be applied!")


In [ ]:
print("=" * 60)
print("TEST 2: Generation Comparison")
print("=" * 60)
print("\nCompare generation using model.generate() vs manual single-token inference.")

import copy

NUM_TOKENS_TO_GENERATE = 20
prompt_text = "The quick brown fox"
prompt_inputs = tokenizer(prompt_text, return_tensors="pt")
prompt_ids = prompt_inputs.input_ids.to(device)

print(f"\nPrompt: '{prompt_text}'")
print(f"Prompt tokens: {prompt_ids.shape[1]}")
print(f"Generating {NUM_TOKENS_TO_GENERATE} tokens...")

with torch.no_grad():
    generated_normal = model.generate(
        prompt_ids,
        max_new_tokens=NUM_TOKENS_TO_GENERATE,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
    )
    
normal_new_tokens = generated_normal[0, prompt_ids.shape[1]:].tolist()

with torch.no_grad():
    past_key_values = None
    for pos in range(prompt_ids.shape[1]):
        current_ids = prompt_ids[:, pos:pos+1]
        outputs = model(current_ids, past_key_values=past_key_values, use_cache=True)
        past_key_values = copy.deepcopy(outputs.past_key_values)
    
    manual_new_tokens = []
    manual_logits_list = []
    
    for _ in range(NUM_TOKENS_TO_GENERATE):
        logits = outputs.logits[:, -1, :]
        next_token = logits.argmax(dim=-1, keepdim=True)
        manual_new_tokens.append(next_token.item())
        manual_logits_list.append(logits.detach().cpu())
        
        outputs = model(next_token, past_key_values=past_key_values, use_cache=True)
        past_key_values = copy.deepcopy(outputs.past_key_values)

print("\n" + "=" * 40)
print("GENERATION RESULTS")
print("=" * 40)

normal_text = tokenizer.decode(normal_new_tokens)
manual_text = tokenizer.decode(manual_new_tokens)

print(f"\nmodel.generate():  '{normal_text}'")
print(f"Manual single-pass: '{manual_text}'")

tokens_match = normal_new_tokens == manual_new_tokens
matching_count = sum(1 for a, b in zip(normal_new_tokens, manual_new_tokens) if a == b)

print(f"\n Token-by-token comparison:")
print(f"  Matching tokens: {matching_count}/{NUM_TOKENS_TO_GENERATE}")

if tokens_match:
    print(f"\n✓ PASS: All generated tokens match exactly!")
else:
    print(f"\n✗ MISMATCH: Generated tokens differ.")
    print(f"\n  Position-by-position:")
    for i, (n, m) in enumerate(zip(normal_new_tokens, manual_new_tokens)):
        status = "✓" if n == m else "✗"
        print(f"    {i}: {status} normal={n} ({tokenizer.decode([n])!r}) vs manual={m} ({tokenizer.decode([m])!r})")
